# 01 · Build the song catalog and the mood model

**Muiso data pipeline.** Traces to FR-3 (scoring inputs), NFR-1 (scores are precomputed), NFR-2 (privacy rules) and the Data Models section of the spec.

**Goal.** Learn a model that predicts a song's mood (valence, arousal) from inputs we can fetch for *any* song, so it can later score tracks from a user's Spotify listening history, not just the songs in MuSe.

| Role | Source | Available for any song? |
|---|---|---|
| **Label** (what we predict) | MuSe valence / arousal (Last.fm tags) | No. MuSe songs only, which is why it is the target |
| Input: lyric emotion | Genius lyrics (via `lyricsgenius`) → emotion classifier | Yes, if the song is on Genius |
| Input: tempo | ReccoBeats, then GetSongBPM as a fallback | Yes, if either API knows the song |
| Input: genre | MuSe genre here; Spotify artist genres at run time | Yes |

**Rules this notebook enforces**
1. Lyric text is cached only in `data/raw/lyrics/` (gitignored). It never goes into a DataFrame that gets saved, a Parquet file, `catalog.db`, or printed cell output.
2. Every API response is cached in `data/raw/`, so re-running the notebook resumes instead of starting over.
3. Each step records a row count in an audit log, saved to `data/interim/audit_log.csv`.
4. API keys are read from `.env` and never printed.

**How to run.** Fill in `.env`, put the MuSe CSV in `data/raw/` (or set the Kaggle keys), then *Run All*. Start with a small `N_SAMPLE`, check the audit tables, then raise it.

## 0 · Setup

In [ ]:
import json
import os
import re
import sqlite3
import subprocess
import sys
import time
from datetime import datetime, timezone
from difflib import SequenceMatcher
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Repo root: MUISO_ROOT if set (for headless runs), else the folder Jupyter was opened in.
ROOT = Path(os.environ.get("MUISO_ROOT", Path.cwd())).resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from tmr.catalog import create_catalog
from tmr.config import GOAL_END_BPM, ROUTES
from tmr.mood import estimate_mood
from tmr.scoring import calm_score

load_dotenv(ROOT / ".env")

RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
MODELS = ROOT / "models"
CACHE = {name: RAW / name for name in ("reccobeats", "getsongbpm", "genius_meta", "lyrics")}
for p in [RAW, INTERIM, MODELS, *CACHE.values()]:
    p.mkdir(parents=True, exist_ok=True)

### Run parameters
Everything you might want to change lives in this cell, so a reviewer can see exactly how a run was configured.

In [ ]:
N_SAMPLE = 2000         # songs to process this run. Genius takes ~1.5 s/song, so 2000 -> ~35 min
                        # (~64% survive the BPM filter). Without Genius, 5000 takes ~2 min.
RANDOM_SEED = 200
MATCH_THRESHOLD = 0.80  # min title AND artist similarity to accept a Genius search result
MIN_LYRIC_WORDS = 20    # fewer words than this after cleaning -> treat as instrumental
BPM_RANGE = (40, 220)   # tempos outside this range are treated as bad data
TOP_GENRES = 12         # genres kept as model inputs; the rest become "other"
MAX_CONSECUTIVE_ERRORS = 10  # stop scraping if Genius starts blocking us

EMOTION_MODEL = "j-hartmann/emotion-english-distilroberta-base"
MUSE_KAGGLE_DATASET = "cakiki/muse-the-musical-sentiment-dataset"
RECCOBEATS_URL = "https://api.reccobeats.com/v1/audio-features"
GETSONGBPM_URL = "https://api.getsongbpm.com/search/"   # check getsongbpm.com/api if this 404s

CATALOG_DB = ROOT / "catalog.db"

In [ ]:
AUDIT = []

def audit(step, n, note=""):
    # Record how many rows survive each step.
    AUDIT.append({"step": step, "rows": int(n), "note": note,
                  "at": datetime.now(timezone.utc).isoformat(timespec="seconds")})
    print(f"[{step}] {n:,} rows  {note}")

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
    except Exception:
        return "unknown"

RUN_INFO = {"started": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "git_commit": git_commit(), "python": sys.version.split()[0],
            "pandas": pd.__version__, "n_sample": N_SAMPLE, "seed": RANDOM_SEED}
KEYS_SET = {k: bool(os.getenv(k)) for k in
            ["GENIUS_ACCESS_TOKEN", "GETSONGBPM_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY"]}
print(json.dumps(RUN_INFO, indent=2))
print("Keys present (values never printed):", KEYS_SET)

## 1 · MuSe (labels)

MuSe gives about 90k songs with `valence_tags`, `arousal_tags` and `dominance_tags`, which are averages of Last.fm tag ratings on the Warriner et al. 1–9 scale (midpoint 5). We rescale them to [-1, 1] with `(x − 5) / 4`. The range check below shows the actual min and max before we trust that assumption.

In [ ]:
MUSE_SEARCH = [RAW, ROOT / "data"]          # data/raw/ preferred; data/ also accepted
found = [f for d in MUSE_SEARCH for f in sorted(d.glob("muse*.csv"))]
if not found:
    if KEYS_SET["KAGGLE_USERNAME"] and KEYS_SET["KAGGLE_KEY"]:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(MUSE_KAGGLE_DATASET, path=str(RAW), unzip=True)
    else:
        raise FileNotFoundError(f"Put the MuSe CSV (muse_v3.csv) in {RAW}, "
                                "or set KAGGLE_USERNAME and KAGGLE_KEY in .env")

    found = sorted(RAW.glob("muse*.csv"))

MUSE_CSV = found[0]
muse_raw = pd.read_csv(MUSE_CSV)
audit("muse_loaded", len(muse_raw), MUSE_CSV.name)
muse_raw.head(3)

In [ ]:
EXPECTED = {"track", "artist", "genre", "spotify_id", "valence_tags", "arousal_tags", "dominance_tags"}
missing = EXPECTED - set(muse_raw.columns)
assert not missing, f"MuSe CSV is missing columns: {missing}"

TAGS = ["valence_tags", "arousal_tags", "dominance_tags"]
muse_raw[TAGS].agg(["min", "mean", "max"]).round(2)

In [ ]:
muse = muse_raw.rename(columns={"track": "title"}).copy()
muse["muse_row"] = muse.index          # stable ID from the CSV row, used as song_id

muse = muse.dropna(subset=["spotify_id", "title", "artist"])
audit("has_spotify_id", len(muse))
muse = muse.drop_duplicates(subset="spotify_id")
audit("dedupe_spotify_id", len(muse))
muse = muse.dropna(subset=TAGS)
audit("has_mood_labels", len(muse))

def to_unit(x):
    return ((x - 5) / 4).clip(-1, 1)

muse["muse_valence"] = to_unit(muse["valence_tags"])
muse["muse_arousal"] = to_unit(muse["arousal_tags"])
muse["muse_dominance"] = to_unit(muse["dominance_tags"])
muse["genre"] = muse["genre"].fillna("unknown").str.lower().str.strip()

muse[["muse_valence", "muse_arousal", "muse_dominance"]].describe().round(3)

### Sample
We shuffle once with a fixed seed and take the first `N_SAMPLE` rows. Raising `N_SAMPLE` later keeps every song from the earlier run, and with the caches only the new songs get fetched.

In [ ]:
songs = (muse.sample(frac=1, random_state=RANDOM_SEED)
             .head(N_SAMPLE)
             .reset_index(drop=True)
             [["muse_row", "spotify_id", "title", "artist", "genre",
               "muse_valence", "muse_arousal", "muse_dominance"]])
audit("sample", len(songs), f"N_SAMPLE={N_SAMPLE}")

## 2 · Tempo (BPM)

ReccoBeats first, looked up by Spotify ID in batches of 40. It also returns audio features (energy, valence, danceability, acousticness and so on), which we keep as model inputs. GetSongBPM is the fallback, searched by title and artist, and only runs when `GETSONGBPM_API_KEY` is set. BPM is fetched before lyrics because songs without a tempo are dropped (BPM is a hard filter in the spec), which saves slow Genius calls.

Run the next cell first. It shows one raw ReccoBeats response, so you can confirm the response format matches what the parser expects.

In [ ]:
def read_cache(folder, key):
    path = CACHE[folder] / f"{key}.json"
    return json.loads(path.read_text()) if path.exists() else None

def write_cache(folder, key, data):
    (CACHE[folder] / f"{key}.json").write_text(json.dumps(data))

probe = requests.get(RECCOBEATS_URL, params={"ids": ",".join(songs.spotify_id.head(2))}, timeout=15)
print("status", probe.status_code)
print(json.dumps(probe.json(), indent=2)[:1500])

In [ ]:
def spotify_id_from(item):
    # ReccoBeats returns its own id; the Spotify id is the last part of the href.
    href = item.get("href") or ""
    return href.rstrip("/").split("/")[-1] if "spotify.com/track/" in href else None

def fetch_reccobeats(ids):
    todo = [i for i in ids if read_cache("reccobeats", i) is None]
    for start in range(0, len(todo), 40):
        batch = todo[start:start + 40]
        for attempt in range(5):
            r = requests.get(RECCOBEATS_URL, params={"ids": ",".join(batch)}, timeout=20)
            if r.status_code != 429:
                break
            time.sleep(2 ** attempt)                          # rate limited: back off and retry
        r.raise_for_status()
        items = r.json().get("content", [])
        found = {spotify_id_from(it): it for it in items}
        for sid in batch:
            write_cache("reccobeats", sid, found.get(sid, {}))   # {} = not found, don't ask again
        time.sleep(0.3)
    return {i: read_cache("reccobeats", i) for i in ids}

recco = fetch_reccobeats(list(songs.spotify_id))
songs["bpm_recco"] = songs.spotify_id.map(lambda i: (recco.get(i) or {}).get("tempo"))
audit("bpm_from_reccobeats", songs.bpm_recco.notna().sum(), "songs with a ReccoBeats tempo")

# ReccoBeats also returns Spotify-style audio features. They exist for any song with a
# Spotify ID, so they are fair model inputs (section 5). Not stored in catalog.db.
AUDIO_FEATURES = ["energy", "valence", "danceability", "acousticness",
                  "instrumentalness", "loudness", "speechiness", "mode"]
for f in AUDIO_FEATURES:
    songs[f"audio_{f}"] = songs.spotify_id.map(lambda i, f=f: (recco.get(i) or {}).get(f))

In [ ]:
def norm(s):
    s = str(s).lower()
    s = re.sub(r"\(.*?\)|\[.*?\]", " ", s)                 # (feat. X), [Remastered]
    s = re.sub(r"\b(feat|ft|featuring)\b.*", " ", s)
    s = re.sub(r"[^a-z0-9 ]+", " ", s)
    return " ".join(s.split())

def similarity(a, b):
    return SequenceMatcher(None, norm(a), norm(b)).ratio()

def fetch_getsongbpm(row):
    cached = read_cache("getsongbpm", row.spotify_id)
    if cached is not None:
        return cached
    r = requests.get(GETSONGBPM_URL, timeout=15, params={
        "api_key": os.environ["GETSONGBPM_API_KEY"], "type": "both",
        "lookup": f"song:{row.title} artist:{row.artist}"})
    r.raise_for_status()
    results = r.json().get("search")
    best = {}
    if isinstance(results, list):                           # {"search": {"error": ...}} when nothing found
        for res in results:
            ts = similarity(row.title, res.get("title", ""))
            ars = similarity(row.artist, (res.get("artist") or {}).get("name", ""))
            if min(ts, ars) >= MATCH_THRESHOLD and res.get("tempo"):
                best = {"tempo": float(res["tempo"]), "title_sim": round(ts, 3), "artist_sim": round(ars, 3)}
                break
    write_cache("getsongbpm", row.spotify_id, best)
    time.sleep(1.2)                                          # stay under the free-tier rate limit
    return best

songs["bpm_gsb"] = np.nan
if KEYS_SET["GETSONGBPM_API_KEY"]:
    need = songs[songs.bpm_recco.isna()]
    for idx, row in need.iterrows():
        songs.at[idx, "bpm_gsb"] = fetch_getsongbpm(row).get("tempo", np.nan)
    audit("bpm_from_getsongbpm", songs.bpm_gsb.notna().sum(), f"of {len(need)} tried")
else:
    print("GETSONGBPM_API_KEY not set; skipping the fallback.")

In [ ]:
songs["bpm"] = songs.bpm_recco.fillna(songs.bpm_gsb)
songs["bpm_source"] = np.where(songs.bpm_recco.notna(), "reccobeats",
                       np.where(songs.bpm_gsb.notna(), "getsongbpm", None))
out_of_range = ~songs.bpm.between(*BPM_RANGE)
songs.loc[out_of_range, ["bpm", "bpm_source"]] = [np.nan, None]
print(songs.bpm_source.value_counts(dropna=False))

songs = songs.dropna(subset=["bpm"]).drop(columns=["bpm_recco", "bpm_gsb"]).reset_index(drop=True)
audit("has_bpm", len(songs), f"BPM in {BPM_RANGE}")

## 3 · Lyrics (Genius via `lyricsgenius`)

`lyricsgenius` searches the Genius API and then scrapes the lyrics from the song page. Because Genius can't search by Spotify ID, we search by title and artist, pick the best of the top 10 hits ourselves, and **accept it only if both title and artist are at least `MATCH_THRESHOLD` similar** to MuSe. Songs that are on Genius with no lyrics are marked `matched_no_lyrics` and treated as instrumentals. Rejected matches are logged and shown in section 7 for spot-checking.

* Lyric text is kept in the in-memory dict `LYRICS` and the gitignored cache. It is never a DataFrame column.
* A network error is not cached, so that song is retried on the next run. After `MAX_CONSECUTIVE_ERRORS` errors in a row the loop stops, which usually means Genius is blocking the IP.
* Genius's terms do not allow scraping. This is noted as a limitation in the report.
* **If `GENIUS_ACCESS_TOKEN` is blank, this section and the emotion classifier are skipped.** Every song is treated as having no lyrics, and the model uses audio features, BPM and genre only. Add the token and re-run to include lyrics.

In [ ]:
USE_LYRICS = KEYS_SET["GENIUS_ACCESS_TOKEN"]
print("Lyrics step:", "ON" if USE_LYRICS else "SKIPPED (no GENIUS_ACCESS_TOKEN)")

if USE_LYRICS:
    from lyricsgenius import Genius
    genius = Genius(os.environ["GENIUS_ACCESS_TOKEN"], timeout=15, retries=2, sleep_time=0.3)

def best_hit(row):
    # Search Genius and return the hit whose title AND artist best match MuSe.
    # We do the matching ourselves instead of genius.search_song(), which returns
    # None for instrumentals and so can't tell "not on Genius" from "no lyrics".
    hits = genius.search_songs(f"{row.title} {row.artist}").get("hits", [])
    best = None
    for h in hits[:10]:
        res = h["result"]
        ts = similarity(row.title, res.get("title", ""))
        ars = similarity(row.artist, (res.get("primary_artist") or {}).get("name", ""))
        if best is None or min(ts, ars) > best[0]:       # strict >: ties keep Genius's ranking
            best = (min(ts, ars), ts, ars, res)
    return best

def fetch_lyrics(row):
    # Return (meta, text). meta has no lyric text and is safe to save.
    #   status: not_found | rejected_match | matched_no_lyrics | matched
    meta_path = CACHE["genius_meta"] / f"{row.spotify_id}.json"
    text_path = CACHE["lyrics"] / f"{row.spotify_id}.txt"
    if meta_path.exists():
        meta = json.loads(meta_path.read_text())
        return meta, (text_path.read_text() if text_path.exists() else None)

    best = best_hit(row)                                       # may raise on network errors
    text = None
    if best is None:
        meta = {"status": "not_found"}
    else:
        score, ts, ars, res = best
        meta = {"genius_id": res["id"], "genius_title": res.get("title"),
                "genius_artist": (res.get("primary_artist") or {}).get("name"),
                "title_sim": round(ts, 3), "artist_sim": round(ars, 3)}
        if score < MATCH_THRESHOLD:
            meta["status"] = "rejected_match"
        elif res.get("instrumental"):
            meta["status"] = "matched_no_lyrics"
        else:
            text = genius.lyrics(song_url=res["url"], remove_section_headers=True)
            meta["status"] = "matched" if text and text.strip() else "matched_no_lyrics"
            if meta["status"] == "matched":
                text_path.write_text(text)
            else:
                text = None
    meta_path.write_text(json.dumps(meta))
    return meta, text

In [ ]:
LYRICS = {}          # spotify_id -> raw lyric text, in memory only
META = {}
errors_in_a_row = 0
t0 = time.time()
for i, row in enumerate(songs.itertuples(index=False) if USE_LYRICS else []):
    try:
        META[row.spotify_id], text = fetch_lyrics(row)
        errors_in_a_row = 0
    except Exception as e:
        META[row.spotify_id], text = {"status": "error", "error": type(e).__name__}, None
        errors_in_a_row += 1
        if errors_in_a_row >= MAX_CONSECUTIVE_ERRORS:
            print(f"Stopping after {errors_in_a_row} errors in a row (last: {type(e).__name__}). Re-run later.")
            break
    if text:
        LYRICS[row.spotify_id] = text
    if (i + 1) % 100 == 0:
        print(f"{i + 1}/{len(songs)}  {time.time() - t0:.0f}s")

songs["genius_status"] = songs.spotify_id.map(
    lambda s: META.get(s, {}).get("status", "not_tried" if USE_LYRICS else "skipped_no_token"))
print(songs.genius_status.value_counts())
audit("genius_matched", songs.genius_status.isin(["matched", "matched_no_lyrics"]).sum(), "incl. instrumentals")

### Clean lyrics, detect instrumentals, keep English

Scraped text usually starts with a line like `"12 ContributorsSong Title Lyrics"` and ends with `"…Embed"`, and may include ad text. Those get stripped. Songs with fewer than `MIN_LYRIC_WORDS` words left are treated as **instrumental**: they keep `lyric_valence = NULL` and stay in the catalog. Songs with lyrics that `langdetect` doesn't detect as English are dropped, since the spec targets English-language songs.

In [ ]:
if LYRICS:
    from langdetect import DetectorFactory, detect
    DetectorFactory.seed = 0

def clean_lyrics(text):
    lines = text.splitlines()
    if lines and ("Contributor" in lines[0] or lines[0].rstrip().endswith("Lyrics")):
        lines = lines[1:]
    text = "\n".join(lines)
    text = re.sub(r"\d*Embed\s*$", "", text.strip())
    text = re.sub(r"You might also like", " ", text)
    text = re.sub(r"See .{0,60} LiveGet tickets as low as \$\d+", " ", text)
    text = re.sub(r"\[.*?\]", " ", text)
    return "\n".join(l.strip() for l in text.splitlines() if l.strip())

CLEAN = {sid: clean_lyrics(t) for sid, t in LYRICS.items()}

def lyric_kind(sid):
    t = CLEAN.get(sid)
    if META.get(sid, {}).get("status") == "matched_no_lyrics":
        return "instrumental"                          # on Genius, but no lyrics
    if t is None:
        return "no_lyrics"
    if len(t.split()) < MIN_LYRIC_WORDS:
        return "instrumental"
    try:
        return "english" if detect(t) == "en" else "non_english"
    except Exception:
        return "unknown_language"

songs["lyric_kind"] = songs.spotify_id.map(lyric_kind)
print(songs.lyric_kind.value_counts())
songs = songs[~songs.lyric_kind.isin(["non_english", "unknown_language"])].reset_index(drop=True)
audit("english_or_instrumental", len(songs), "dropped non-English lyrics")

## 4 · Lyric emotion scores

**Classifier:** `j-hartmann/emotion-english-distilroberta-base`, which gives probabilities for 7 emotions. Lyrics are split into 200-word chunks (the model reads at most 512 tokens) and the chunk probabilities are averaged.

**From emotions to (valence, arousal):** each emotion gets a fixed point on the circumplex below, and the song's lyric (V, A) is the probability-weighted average of those points. This mapping is a modelling assumption and is kept in this cell so it can be reviewed. The model in section 5 also gets the 7 raw probabilities, so it can learn around a poor mapping.

**Theme flags:** a short keyword list for `self_harm` and `substance`. It's crude on purpose and easy to audit, and it's a candidate for replacement. Only the flag names are stored, never the matched words.

In [ ]:
EMOTION_VA = {   # (valence, arousal) for each classifier label
    "joy":      ( 0.8,  0.5),
    "surprise": ( 0.3,  0.7),
    "neutral":  ( 0.0,  0.0),
    "sadness":  (-0.7, -0.4),
    "fear":     (-0.6,  0.6),
    "anger":    (-0.6,  0.8),
    "disgust":  (-0.7,  0.3),
}
EMOTIONS = list(EMOTION_VA)

NEED_CLASSIFIER = bool((songs.lyric_kind == "english").any())
if NEED_CLASSIFIER:
    from transformers import pipeline
    clf = pipeline("text-classification", model=EMOTION_MODEL, top_k=None, truncation=True)
else:
    print("No English lyrics to score; classifier not loaded.")

def chunks(text, n_words=200):
    w = text.split()
    return [" ".join(w[i:i + n_words]) for i in range(0, len(w), n_words)] or [""]

def emotion_probs(text):
    outs = clf(chunks(text))
    return pd.DataFrame([{d["label"]: d["score"] for d in o} for o in outs]).mean()[EMOTIONS]

def lyric_va(probs):
    v = sum(probs[e] * EMOTION_VA[e][0] for e in EMOTIONS)
    a = sum(probs[e] * EMOTION_VA[e][1] for e in EMOTIONS)
    return float(np.clip(v, -1, 1)), float(np.clip(a, -1, 1))

# Sanity check on made-up sentences (not lyrics) before trusting it on songs.
for s in [] if not NEED_CLASSIFIER else ["I feel so happy and free, dancing in the sun",
          "I'm alone and crying, everything is gone",
          "I'm so angry I could scream and break everything"]:
    p = emotion_probs(s)
    print(f"{p.idxmax():8s}  V,A={lyric_va(p)}  <- {s}")

In [ ]:
FLAG_PATTERNS = {
    "self_harm": [r"\bkill(ing)? myself\b", r"\bsuicid\w*", r"\bcut(ting)? myself\b",
                  r"\bwant(ed)? to die\b", r"\bend it all\b", r"\bslit (my|your) wrists?\b"],
    "substance": [r"\bcocaine\b", r"\bheroin\b", r"\boverdos\w*", r"\bxanax\b",
                  r"\bmolly\b", r"\bpercocets?\b", r"\bget(ting)? high\b", r"\bdrunk\b"],
}

def theme_flags(text):
    t = text.lower()
    return ",".join(name for name, pats in FLAG_PATTERNS.items()
                    if any(re.search(p, t) for p in pats))

rows = []
for sid in songs.loc[songs.lyric_kind == "english", "spotify_id"]:
    p = emotion_probs(CLEAN[sid])
    v, a = lyric_va(p)
    rows.append({"spotify_id": sid, "lyric_valence": v, "lyric_arousal": a,
                 "flags": theme_flags(CLEAN[sid]), **{f"emo_{e}": p[e] for e in EMOTIONS}})

lyric_scores = pd.DataFrame(rows, columns=["spotify_id", "lyric_valence", "lyric_arousal", "flags",
                                           *[f"emo_{e}" for e in EMOTIONS]])
songs = songs.merge(lyric_scores, on="spotify_id", how="left")
songs["flags"] = songs["flags"].fillna("")
audit("lyrics_scored", songs.lyric_valence.notna().sum())
print("flag counts:", songs["flags"].str.split(",").explode().replace("", np.nan).value_counts().to_dict())

## 5 · Mood model: predict MuSe (V, A) from inputs available for any song

**Inputs:** ReccoBeats audio features (energy, valence, danceability, acousticness, instrumentalness, loudness, speechiness, mode), 7 lyric emotion probabilities and lyric V/A (0 plus a `has_lyrics = 0` indicator when there are no lyrics), BPM, and top-`TOP_GENRES` genre one-hot.
**Targets:** `muse_valence`, `muse_arousal`. MuSe is never an input, since it won't exist for listening-history songs.

Two approaches are compared against MuSe on the same held-out songs:

1. **Rules (deterministic):** `tmr.mood.estimate_mood()`, a fixed formula whose directions come from the music-psychology literature (citations in `src/tmr/mood.py`). No fitting, so its score is an honest out-of-sample number.
2. **Learned:** the models below, fit to MuSe.

`MOOD_METHOD` (set after the comparison) decides which one fills `est_valence` / `est_arousal`. The default is the rules, because they are explainable, citable and behave the same on any song. The learned model is still saved for comparison.

For the learned models, we compare three on an 80/20 held-out split: **predict-the-mean** (baseline), **ridge**, and **gradient boosting**. The best one is refit on all songs and saved. For catalog songs, `est_valence` / `est_arousal` come from **5-fold out-of-fold predictions**, so catalog songs and new listening-history songs are scored the same way, and no song is scored by a model that saw its own label.

`build_features()` is the exact function that inference will need, so it has to move to `src/tmr/` when the app is built.

In [ ]:
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_predict, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

GENRES = songs.genre.value_counts().head(TOP_GENRES).index.tolist()

def build_features(df, genres=GENRES):
    X = pd.DataFrame(index=df.index)
    X["has_lyrics"] = df["lyric_valence"].notna().astype(int)
    for col in ["lyric_valence", "lyric_arousal", *[f"emo_{e}" for e in EMOTIONS]]:
        X[col] = df[col].fillna(0.0)
    X["has_audio"] = df["audio_energy"].notna().astype(int)
    for f in AUDIO_FEATURES:
        X[f"audio_{f}"] = df[f"audio_{f}"].astype(float).fillna(0.0)
    X["bpm"] = df["bpm"]
    g = df["genre"].where(df["genre"].isin(genres), "other")
    for name in [*genres, "other"]:
        X[f"genre_{name}"] = (g == name).astype(int)
    return X

X = build_features(songs)
y = songs[["muse_valence", "muse_arousal"]].to_numpy()
print("features:", X.shape[1], "songs:", len(X))

In [ ]:
CANDIDATES = {
    "baseline_mean": DummyRegressor(strategy="mean"),
    "ridge": make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    "grad_boost": MultiOutputRegressor(HistGradientBoostingRegressor(
        max_iter=200, learning_rate=0.05, max_depth=4, random_state=RANDOM_SEED)),
}

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
results = []
for name, model in CANDIDATES.items():
    pred = model.fit(X_tr, y_tr).predict(X_te)
    for j, target in enumerate(["valence", "arousal"]):
        results.append({"model": name, "target": target,
                        "MAE": mean_absolute_error(y_te[:, j], pred[:, j]),
                        "R2": r2_score(y_te[:, j], pred[:, j]),
                        "spearman": pd.Series(y_te[:, j]).corr(pd.Series(pred[:, j]), method="spearman")})
# Rules: no fitting, so we score them on the same held-out songs.
rule = pd.DataFrame([estimate_mood({**r._asdict(), **{f: getattr(r, f"audio_{f}") for f in AUDIO_FEATURES},
                                    "lyric_valence": None if pd.isna(r.lyric_valence) else r.lyric_valence,
                                    "lyric_arousal": None if pd.isna(r.lyric_arousal) else r.lyric_arousal})
                     for r in songs.itertuples()], columns=["rule_valence", "rule_arousal"], index=songs.index)
songs[["rule_valence", "rule_arousal"]] = rule
rule_te = rule.loc[X_te.index].to_numpy(dtype=float)
ok = ~np.isnan(rule_te).any(axis=1)
for j, target in enumerate(["valence", "arousal"]):
    results.append({"model": "rules", "target": target,
                    "MAE": mean_absolute_error(y_te[ok, j], rule_te[ok, j]),
                    "R2": r2_score(y_te[ok, j], rule_te[ok, j]),
                    "spearman": pd.Series(y_te[ok, j]).corr(pd.Series(rule_te[ok, j]), method="spearman")})
results = pd.DataFrame(results).round(3)
results.pivot(index="model", columns="target", values=["MAE", "R2", "spearman"])

In [ ]:
# "rules" = deterministic literature-based formula; "learned" = best fitted model.
# Rule of thumb: only switch to "learned" if it is clearly better (mean held-out
# Spearman at least 0.10 higher) AND explains a real share of variance (mean R2 >= 0.2).
MOOD_METHOD = "rules"

best_name = (results[~results.model.isin(["baseline_mean", "rules"])]
             .groupby("model").MAE.mean().idxmin())
sp = results.groupby("model").spearman.mean()
r2 = results.groupby("model").R2.mean()
learned_clearly_better = (sp[best_name] - sp["rules"] >= 0.10) and (r2[best_name] >= 0.2)
print(f"best learned model: {best_name}  (mean Spearman {sp[best_name]:.3f}, mean R2 {r2[best_name]:.3f})")
print(f"rules:                          (mean Spearman {sp['rules']:.3f}, mean R2 {r2['rules']:.3f})")
print("learned clearly better by the rule of thumb:", learned_clearly_better)
print("MOOD_METHOD =", MOOD_METHOD)

oof = cross_val_predict(CANDIDATES[best_name], X, y,
                        cv=KFold(5, shuffle=True, random_state=RANDOM_SEED))
songs["learned_valence"] = np.clip(oof[:, 0], -1, 1)
songs["learned_arousal"] = np.clip(oof[:, 1], -1, 1)
src = "rule" if MOOD_METHOD == "rules" else "learned"
songs["est_valence"] = songs[f"{src}_valence"]
songs["est_arousal"] = songs[f"{src}_arousal"]

final_model = CANDIDATES[best_name].fit(X, y)
bundle = {"model": final_model, "feature_columns": list(X.columns), "genres": GENRES,
          "emotion_model": EMOTION_MODEL, "emotion_va": EMOTION_VA, "run_info": RUN_INFO,
          "holdout_metrics": results.to_dict("records")}
joblib.dump(bundle, MODELS / "mood_model.joblib")
print("saved", MODELS / "mood_model.joblib")

## 5b · Can the catalog serve each route?

For each goal in the Routes table, count songs near the **end mood** (within 0.35 on the mood grid) that also pass that goal's end-stage BPM filter (`GOAL_END_BPM` in `src/tmr/config.py`, with citations). A goal with very few songs here can't finish its route well. Start moods aren't checked, because they vary by user.

In [ ]:
coverage = []
for goal, route in ROUTES.items():
    ev, ea = route["end"]
    near = np.hypot(songs.est_valence - ev, songs.est_arousal - ea) <= 0.35
    lo, hi = GOAL_END_BPM[goal]
    in_bpm = songs.bpm.between(lo, hi)
    coverage.append({"goal": goal, "end_mood": route["end"], "bpm_range": (lo, hi),
                     "near_end_mood": int(near.sum()), "near_and_bpm_ok": int((near & in_bpm).sum())})
pd.DataFrame(coverage)

## 6 · Calm score, privacy check, write outputs

`calm_score` is imported from `src/tmr/scoring.py`, the same function the app and UT-2 use. BPM enters **only** through its tempo term. It's also a model input for `est_arousal`, so there is some double counting. Section 7 reports the correlation so the team can decide whether to drop BPM from the model (spec: "BPM must be used as a single input").

In [ ]:
songs["calm_score"] = [
    calm_score({"bpm": r.bpm, "est_arousal": r.est_arousal,
                "lyric_valence": None if pd.isna(r.lyric_valence) else r.lyric_valence})
    for r in songs.itertuples()]

SONG_COLS = ["song_id", "spotify_id", "title", "genre", "artist",
             "muse_dominance", "muse_valence", "muse_arousal", "bpm", "bpm_source",
             "lyric_valence", "lyric_arousal", "flags", "est_valence", "est_arousal", "calm_score"]
catalog = songs.rename(columns={"muse_row": "song_id"})[SONG_COLS]

# Privacy guard: nothing that could hold lyric text leaves this notebook.
assert not {"lyrics", "lyric_text", "text"} & set(catalog.columns)
for col in catalog.select_dtypes("object"):
    longest = catalog[col].astype(str).str.len().max()
    assert longest < 300, f"{col} has a {longest}-char value; looks like it could be lyric text"
lyric_set = set(CLEAN.values())
assert not any(v in lyric_set for col in catalog.select_dtypes("object") for v in catalog[col])
print("privacy checks passed")

In [ ]:
catalog.to_parquet(INTERIM / "songs.parquet", index=False)
X.assign(spotify_id=songs.spotify_id.values).to_parquet(INTERIM / "model_features.parquet", index=False)

match_report = pd.DataFrame([{"spotify_id": sid, **m} for sid, m in META.items()],
                            columns=["spotify_id", "status", "genius_id", "genius_title", "genius_artist",
                                     "title_sim", "artist_sim", "error"])
match_report.to_parquet(INTERIM / "genius_match_report.parquet", index=False)

conn = create_catalog(str(CATALOG_DB))
conn.execute("DELETE FROM songs")
catalog.to_sql("songs", conn, if_exists="append", index=False)
conn.commit()
n_db = conn.execute("SELECT COUNT(*) FROM songs").fetchone()[0]
conn.close()
audit("catalog_db_rows", n_db, str(CATALOG_DB.name))

## 7 · Audit report

In [ ]:
audit_df = pd.DataFrame(AUDIT)
audit_df.to_csv(INTERIM / "audit_log.csv", index=False)
(INTERIM / "run_info.json").write_text(json.dumps({**RUN_INFO, "best_model": best_name,
    "finished": datetime.now(timezone.utc).isoformat(timespec="seconds")}, indent=2))
audit_df[["step", "rows", "note"]]

In [ ]:
print("Usable songs (report this number):", len(catalog))
print("\nBPM source:\n", catalog.bpm_source.value_counts())
print("\nLyrics:\n", songs.lyric_kind.value_counts())
print("\nDouble-counting check, correlation with est_arousal:",
      {"bpm": round(catalog.bpm.corr(catalog.est_arousal), 3),
       "audio_energy": round(songs.audio_energy.astype(float).corr(songs.est_arousal), 3)})
catalog[["est_valence", "est_arousal", "calm_score"]].describe().round(3)

In [ ]:
# Spot-check: Genius results we rejected. Titles and artists only, no lyrics.
rej = match_report[match_report.status == "rejected_match"]
print(f"{len(rej)} rejected matches")
rej.merge(songs[["spotify_id", "title", "artist"]], on="spotify_id", how="left") \
   [["title", "artist", "genius_title", "genius_artist", "title_sim", "artist_sim"]].head(15)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(catalog.muse_valence, catalog.est_valence, s=6, alpha=0.4)
axes[0].plot([-1, 1], [-1, 1], "k--", lw=1)
axes[0].set(xlabel="MuSe valence (label)", ylabel="predicted valence (out-of-fold)",
            title="Valence: predicted vs label")
sc = axes[1].scatter(catalog.est_valence, catalog.est_arousal, c=catalog.calm_score,
                     cmap="viridis", s=6, alpha=0.6)
axes[1].set(xlabel="est_valence", ylabel="est_arousal", xlim=(-1, 1), ylim=(-1, 1),
            title="Catalog on the mood grid (color = calm_score)")
fig.colorbar(sc, ax=axes[1], label="calm_score")
plt.tight_layout()

## Next steps
* `est_valence` / `est_arousal` come from `tmr.mood.estimate_mood()` (rules), which the app can call directly on listening-history songs. To score lyrics at run time, `clean_lyrics`, `emotion_probs` and `lyric_va` still need to move into `src/tmr/`.
* Wire `GOAL_END_BPM` into the planner as the settle-stage filter.
* The rules leave BPM out of `est_arousal`, so BPM counts once (via `calm_score` and the BPM filter). The learned model still uses it; see the double-counting check above.
* Replace the keyword `FLAG_PATTERNS` with something better if the team decides flags matter for filtering.
* Raise `N_SAMPLE` once the audit numbers look right. The model needs a few thousand songs to be worth much.